# 🚀 Lending Club - Pipeline Completo (Baselines, PyTorch Deep Learning & Relatório PDF)
Este notebook unifica todo o fluxo experimental do projeto:
1. **Verificação de GPU e Configuração do Repositório**
2. **Download do Dataset / Fallback com Polars**
3. **Baselines Clássicas (Scikit-Learn)**: Regressão Logística e Regressão Linear
4. **Modelos de Deep Learning (PyTorch MLP em GPU)**:
   - **Parte A**: Classificação Binária (`loan_status`) com He Initialization, LayerNorm e Dropout.
   - **Parte B**: Regressão Contínua (`int_rate`) com Xavier Initialization e ReduceLROnPlateau.
5. **Exibição de Gráficos e Curvas de Perda**
6. **Geração Automática do Relatório Técnico Executivo em PDF**

In [ ]:
# 1. Verificar aceleração GPU no PyTorch
import torch

print(f"Versão do PyTorch: {torch.__version__}")
print(f"GPU Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU Ativo: {torch.cuda.get_device_name(0)}")
        {
            "cell_type": "code",
            "id": "#VSC-23889072",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 2. Clonar o repositório modular e instalar dependências (Google Colab)",
                "import os",
                "",
                "REPO_URL = \"https://github.com/anacaetano02/Projetos-Redes-Neurais.git\"",
                "",
                "# Se o diretório `src` não existir, clona o repositório na raiz do notebook",
                "if not os.path.exists('src'):",
                "    print('Clonando repositório...')",
                "    !git clone {REPO_URL} .",
                "    print('Instalando dependências...')",
                "    !pip install -r requirements.txt",
                "else:",
                "    print('Repositório já presente — pulando clone. Instalando dependências...')",
                "    !pip install -r requirements.txt"
            ]
        },
from src.data_pipeline import download_kaggle_dataset, pipeline_dados_lending_club_completo

# Download via Kaggle CLI ou fallback para base simulada
csv_path = download_kaggle_dataset()

SEED = 42
df_train, df_val, df_test = pipeline_dados_lending_club_completo(csv_path, seed=SEED)

## 📊 1. Módulo de Baselines Clássicas (Scikit-Learn)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, confusion_matrix, r2_score, mean_absolute_error, mean_squared_error

# Isolando matrizes de características e respostas
exclude_cols = ["classification_target", "int_rate", "loan_status"]
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X_train = df_train.select(feature_cols).to_numpy()
y_train_class = df_train["classification_target"].to_numpy()
y_train_reg = df_train["int_rate"].to_numpy()

X_test = df_test.select(feature_cols).to_numpy()
y_test_class = df_test["classification_target"].to_numpy()
y_test_reg = df_test["int_rate"].to_numpy()

print("="*60)
print("     BASELINE A: CLASSIFICAÇÃO COM REGRESSÃO LOGÍSTICA")
print("="*60)
clf = LogisticRegression(random_state=SEED, max_iter=1000)
clf.fit(X_train, y_train_class)
preds_class = clf.predict(X_test)

print("\nMatriz de Confusão:")
print(confusion_matrix(y_test_class, preds_class))
print("\nRelatório Completo de Métricas (Teste):")
print(classification_report(y_test_class, preds_class, target_names=["Fully Paid (0)", "Charged Off (1)"]))

print("\n" + "="*60)
print("     BASELINE B: REGRESSÃO COM REGRESSÃO LINEAR MULTIVARIADA")
print("="*60)
reg = LinearRegression()
reg.fit(X_train, y_train_reg)
preds_reg = reg.predict(X_test)

mae = mean_absolute_error(y_test_reg, preds_reg)
mse = mean_squared_error(y_test_reg, preds_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, preds_reg)

print(f"\nMétricas do Ajuste Linear Clássico (Teste):")
print(f"Mean Absolute Error (MAE):      {mae:.6f}")
print(f"Mean Squared Error (MSE):       {mse:.6f}")
print(f"Root Mean Squared Error (RMSE):  {rmse:.6f}")
print(f"Coeficiente R² (Determinação):   {r2:.4f}")
print("="*60)

## 🧠 2. Experimento PyTorch A: Classificação Binária (MLP em GPU)

In [ ]:
from functools import partial
import torch.nn as nn
from torch.utils.data import DataLoader

from src.utils import set_seed, init_weights, EarlyStopping
from src.dataset import LendingClubDataset
from src.models import LendingClubMLP
from src.trainer import train_one_epoch, validate_one_epoch, evaluate_on_test_set
from src.visualization import plotar_curvas_de_perda

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de execução ativo: {device}\n")

# Datasets e DataLoaders de Classificação
ds_train_class = LendingClubDataset(df_train, mode="classification")
ds_val_class = LendingClubDataset(df_val, mode="classification")
ds_test_class = LendingClubDataset(df_test, mode="classification")

train_loader_class = DataLoader(ds_train_class, batch_size=128, shuffle=True)
val_loader_class = DataLoader(ds_val_class, batch_size=256, shuffle=False)
test_loader_class = DataLoader(ds_test_class, batch_size=256, shuffle=False)

input_dim = ds_train_class.features.shape[1]

# Modelo MLP com LayerNorm e Dropout
model_class = LendingClubMLP(
    input_size=input_dim, 
    hidden_size=64, 
    num_classes=2, 
    activation_type="relu",
    dropout_rate=0.3,
    use_norm="layer"
).to(device)

model_class.apply(partial(init_weights, init_type='kaiming_normal', activation_type='relu'))

criterion_class = nn.CrossEntropyLoss()
optimizer_class = torch.optim.Adam(model_class.parameters(), lr=0.005, weight_decay=1e-4)

checkpoint_class_path = 'checkpoints/best_model_classification.pt'
early_stopping_class = EarlyStopping(patience=4, checkpoint_path=checkpoint_class_path, verbose=True)

EPOCHS = 15
historico_class = []
for epoch in range(1, EPOCHS + 1):
    t_metrics = train_one_epoch(model_class, train_loader_class, criterion_class, optimizer_class, device, mode="classification")
    v_metrics = validate_one_epoch(model_class, val_loader_class, criterion_class, device, mode="classification")
    
    print(f"Época {epoch:02d}/{EPOCHS:02d} | Perda Treino: {t_metrics['loss']:.4f} | Perda Val: {v_metrics['loss']:.4f} | Grad Norm: {t_metrics['grad_norm']:.4f}")
    historico_class.append({"train_loss": t_metrics["loss"], "val_loss": v_metrics["loss"]})
    
    early_stopping_class(v_metrics["loss"], model_class)
    if early_stopping_class.early_stop:
        print("Early stopping acionado!")
        break

# Avaliação Final no Teste Cego
model_class.load_state_dict(torch.load(checkpoint_class_path))
evaluate_on_test_set(model_class, test_loader_class, device, mode="classification")
plotar_curvas_de_perda(historico_class, tarefa="classification", caminho_salvamento="curvas_perda_classificacao.png")

## 📈 3. Experimento PyTorch B: Regressão Contínua (MLP em GPU)

In [ ]:
set_seed(SEED)

# Datasets e DataLoaders de Regressão
ds_train_reg = LendingClubDataset(df_train, mode="regression")
ds_val_reg = LendingClubDataset(df_val, mode="regression")
ds_test_reg = LendingClubDataset(df_test, mode="regression")

train_loader_reg = DataLoader(ds_train_reg, batch_size=128, shuffle=True)
val_loader_reg = DataLoader(ds_val_reg, batch_size=256, shuffle=False)
test_loader_reg = DataLoader(ds_test_reg, batch_size=256, shuffle=False)

# Modelo MLP de Regressão com Xavier Normal
model_reg = LendingClubMLP(
    input_size=input_dim, 
    hidden_size=64, 
    num_classes=1, 
    activation_type="tanh",
    dropout_rate=0.2,
    use_norm="layer"
).to(device)

model_reg.apply(partial(init_weights, init_type='xavier_normal', activation_type='tanh'))

criterion_reg = nn.MSELoss()
optimizer_reg = torch.optim.Adam(model_reg.parameters(), lr=0.005, weight_decay=1e-4)
scheduler_reg = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_reg, mode='min', factor=0.5, patience=2)

checkpoint_reg_path = 'checkpoints/best_model_regression.pt'
early_stopping_reg = EarlyStopping(patience=4, checkpoint_path=checkpoint_reg_path, verbose=True)

historico_reg = []
for epoch in range(1, EPOCHS + 1):
    t_metrics = train_one_epoch(model_reg, train_loader_reg, criterion_reg, optimizer_reg, device, mode="regression")
    v_metrics = validate_one_epoch(model_reg, val_loader_reg, criterion_reg, device, mode="regression")
    
    current_lr = optimizer_reg.param_groups[0]['lr']
    print(f"Época {epoch:02d}/{EPOCHS:02d} | LR: {current_lr:.6f} | MSE Treino: {t_metrics['loss']:.6f} | MSE Val: {v_metrics['loss']:.6f}")
    historico_reg.append({"train_loss": t_metrics["loss"], "val_loss": v_metrics["loss"]})
    
    scheduler_reg.step(v_metrics["loss"])
    early_stopping_reg(v_metrics["loss"], model_reg)
    if early_stopping_reg.early_stop:
        print("Early stopping acionado!")
        break

# Avaliação Final no Teste Cego
model_reg.load_state_dict(torch.load(checkpoint_reg_path))
evaluate_on_test_set(model_reg, test_loader_reg, device, mode="regression")
plotar_curvas_de_perda(historico_reg, tarefa="regression", caminho_salvamento="curvas_perda_regressao.png")

## 🖼️ 4. Exibição dos Gráficos Gerados

In [ ]:
from IPython.display import Image, display

print("--- Curvas de Perda: Classificação Binária ---")
display(Image("curvas_perda_classificacao.png"))

print("\n--- Curvas de Perda: Regressão Contínua ---")
display(Image("curvas_perda_regressao.png"))

## 📄 5. Geração Automática do Relatório Técnico em PDF
Compila o relatório formal em PDF combinando métricas, tabelas e gráficos.

In [ ]:
from src.relatorio import generate_charts, build_pdf

print("Gerando gráficos para a documentação...")
generate_charts()

print("Compilando o Relatório Técnico PDF...")
build_pdf()
print("Relatório PDF gerado com sucesso!")